# Module 02 — The Corpus (notebook)

Walkthrough of [`corpus.py`](corpus.py). We'll see what FineWeb-Edu documents look like, what each filter knob does, how fast the stream is, and verify that distributed sharding works.

**Compute:** CPU-only. Streams from HuggingFace's CDN — needs internet but no GPU.

**Time:** ~10 minutes. The first cell to hit the dataset takes a bit longer (initial connection); after that it's smooth.

If you haven't already, log in to HuggingFace from a terminal:
```bash
huggingface-cli login
```
FineWeb-Edu is public and doesn't strictly need a token, but the `datasets` library is happier with one cached.

In [ ]:
from corpus import stream_fineweb_edu, measure_throughput
from itertools import islice
from collections import Counter

## 1. What does a document look like?

Stream one document and inspect every field. The schema is the same for every doc — once you've seen one, you've seen the shape of the whole corpus.

In [ ]:
stream = stream_fineweb_edu(rank=0, world_size=1, shuffle_buffer=100)
doc = next(iter(stream))

for key, value in doc.items():
    if key == "text":
        print(f"{key:18s} (len={len(value)}) {value[:120]!r}...")
    else:
        print(f"{key:18s} {value!r}")

Notice:
- `score` is the educational-quality classifier output (0–5). Our default `min_score=3.0` already cut everything below the middle.
- `language_score` is a separate signal — the language-detection model's confidence. Usually ~1.0 for English.
- `url` and `dump` (the Common Crawl dump) give you traceability back to provenance.
- `token_count` is a pre-computed estimate (using FineWeb-Edu's tokenizer, not ours). Useful for back-of-envelope token-budget math.

Now read the document itself:

In [ ]:
print(doc["text"][:1500])

## 2. What the quality filter throws away

FineWeb-Edu's score is 0–5: 0 is essentially noise (SEO spam, navigation, error pages), 5 is high-information educational content. Compare two streams: a permissive one (`min_score=0`) and a strict one (`min_score=4`).

In [ ]:
def score_distribution(min_score, n=500):
    stream = stream_fineweb_edu(min_score=min_score, min_chars=0, shuffle_buffer=100)
    scores = [round(d["score"]) for d in islice(stream, n)]
    return Counter(scores)

print("min_score=0 (everything):       ", dict(sorted(score_distribution(0).items())))
print("min_score=3 (our default):      ", dict(sorted(score_distribution(3).items())))
print("min_score=4 (strict):           ", dict(sorted(score_distribution(4).items())))

**Takeaway:** Raising `min_score` from 3 to 4 retains roughly 10–20% of what `min_score=3` keeps. You get cleaner data at the cost of less data. For our small course-scale runs, `min_score=3` is the right balance. At larger scale, the tradeoff shifts — billion-parameter pretrains care more about token volume than the last 5% of quality.

## 3. Is the stream fast enough?

Rule of thumb: a loaded A100 wants ~1–2 GB/sec of tokens. Tokens are ~5 bytes each, so call it ~10–20 GB/sec of *raw text* hitting the tokenizer (the tokenizer compresses ~4×).

On a typical cloud pod with decent bandwidth, FineWeb-Edu's streaming should comfortably hit hundreds of MB/sec of raw text. On your laptop over a home connection, expect ~10–50 MB/sec. The numbers we care about:

In [ ]:
stream = stream_fineweb_edu(shuffle_buffer=1000)
stats = measure_throughput(stream, n_docs=500)
for k, v in stats.items():
    print(f"{k:14s} {v:.2f}" if isinstance(v, float) else f"{k:14s} {v}")

If `mb_per_s` is *below* ~10 on a cloud pod, something is wrong — likely a small shuffle buffer, only one dataloader worker, or a slow region. On a laptop it's normal. We will use `torch.utils.data.DataLoader(..., num_workers=N)` in Module 11 to parallelize the network fetch and keep the GPU fed.

## 4. The shuffle buffer matters

Without buffered shuffling, you'd see documents in source-file order — which means hundreds of consecutive docs from the same Common Crawl dump, often the same domain. That's a terrible training signal.

We compare URL diversity in the first 100 docs at two buffer sizes:

In [ ]:
def url_diversity(shuffle_buffer, n=100):
    stream = stream_fineweb_edu(shuffle_buffer=shuffle_buffer)
    domains = [d["url"].split("/")[2] if "://" in d["url"] else d["url"]
               for d in islice(stream, n)]
    return len(set(domains)), len(domains)

for buf in (1, 100, 10_000):
    uniq, total = url_diversity(buf)
    print(f"buffer={buf:>6}  {uniq}/{total} unique domains in first {total} docs")

Bigger buffer → more domain diversity in any given window of docs. The training loop sees a smoother distribution. The cost is memory (the buffer holds that many fully-loaded documents).

## 5. Distributed sharding does the right thing

Two ranks in a `world_size=2` setup must see *disjoint* documents — otherwise your effective batch size is a lie. Verify by collecting IDs from each rank and checking the intersection is empty.

In [ ]:
def ids_from_rank(rank, world_size=2, n=200):
    stream = stream_fineweb_edu(rank=rank, world_size=world_size, shuffle_buffer=100)
    return {d["id"] for d in islice(stream, n)}

rank0_ids = ids_from_rank(0)
rank1_ids = ids_from_rank(1)

print(f"rank 0 ids: {len(rank0_ids)}")
print(f"rank 1 ids: {len(rank1_ids)}")
print(f"overlap:    {len(rank0_ids & rank1_ids)} (should be 0)")

Zero overlap. That's the property every distributed training run relies on.

## Recap

You now have:
- A streaming pipeline that produces FineWeb-Edu documents, filtered, shuffled, and rank-sharded.
- A way to measure whether the stream will bottleneck the GPU.
- An understanding of the knobs (`min_score`, `min_chars`, `shuffle_buffer`) and the tradeoffs they encode.

Next: [Module 03 — Tokenization](../03-tokenization/). The documents become tokens — and the tokenizer choice quietly constrains every architectural decision after it.